# 2.7 RL 알고리즘

REINFORCE, PPO, DPO, GRPO는 위의 개념들을 구체적인 알고리즘으로 구현한다.

각 알고리즘은 서로 다른 안정성-효율성 트레이드오프를 제공한다.

### REINFORCE: 기본 정책 그래디언트
∇J = E[∇log π(a|s) × R] (높은 분산)

### PPO: 근접 정책 최적화  
KL 발산을 제어하여 안정화 (실무 권장)

### DPO: 직접 선호도 최적화
보상 모델 없이 선호도 쌍으로 직접 학습

### GRPO: 그룹 상대 정책
그룹 내 상대적 성과로 평가

In [6]:
from utils_openai import *
import numpy as np

## 실습 1: REINFORCE - 최고 응답으로 프롬프트 개선

In [7]:
heading("REINFORCE: 최고 응답 기반 학습")

prompt = "AI의 윤리적 쟁점을 설명해주세요다"

# 여러 응답 샘플링
responses = []
rewards = []
for i in range(3):
    resp = ask(prompt, temperature=0.7)
    reward = llm_reward(resp, criteria="포괄성과 균형성", max_score=10)
    responses.append(resp)
    rewards.append(reward)
    step_print(i+1, f"Sample {i+1}", f"보상: {reward:.1f}")

best_idx = np.argmax(rewards)
best_resp = responses[best_idx]
step_print(4, "선택", f"최고 보상: {best_resp[:60]}...")

# 프롬프트 개선
improved_prompt = f"{prompt}(예시: {best_resp[:50]}...)" 
print(f"개선된 프롬프트: {improved_prompt[:80]}...")
print("✓ REINFORCE는 최고 응답으로 정책을 개선한다.")


────────────────────────────────────────
  REINFORCE: 최고 응답 기반 학습
────────────────────────────────────────
  [Step 1] Sample 1
    보상: 8.0
  [Step 2] Sample 2
    보상: 8.0
  [Step 3] Sample 3
    보상: 8.0
  [Step 4] 선택
    최고 보상: AI의 윤리적 쟁점은 다양한 측면에서 논의되고 있으며, 다음과 같은 주요 주제들이 있습니다:
    
    1. **편향...
개선된 프롬프트: AI의 윤리적 쟁점을 설명해주세요다(예시: AI의 윤리적 쟁점은 다양한 측면에서 논의되고 있으며, 다음과 같은 주요 주제들이 있습니다...)...
✓ REINFORCE는 최고 응답으로 정책을 개선한다.


## 실습 2: PPO - 안정적인 정책 업데이트

In [8]:
heading("PPO: 클리핑으로 안정화")

# 이전 정책
old_prompt = "좋은 설명을 원해요다"

# 새 정책 (약간 다름)
new_prompt = "좋은 설명을 원해요다 그리고 구체적인 예시도 포함해주세요다"

# KL 발산 추정 (간단한 예시)
kl_div = len(set(old_prompt.split()) ^ set(new_prompt.split())) / len(set(old_prompt.split()) | set(new_prompt.split()))
kl_threshold = 0.2

step_print(1, "이전 정책", old_prompt)
step_print(2, "새 정책", new_prompt)
step_print(3, "KL 발산", f"{kl_div:.3f} (임계값: {kl_threshold})")

if kl_div > kl_threshold:
    print("KL 발산이 크므로 업데이트를 억제한다.")
    final_prompt = old_prompt
else:
    final_prompt = new_prompt

kv_print([("최종 정책", final_prompt)])
print("✓ PPO는 정책 변화를 제어하여 안정성을 보장한다.")



────────────────────────────────────────
  PPO: 클리핑으로 안정화
────────────────────────────────────────
  [Step 1] 이전 정책
    좋은 설명을 원해요다
  [Step 2] 새 정책
    좋은 설명을 원해요다 그리고 구체적인 예시도 포함해주세요다
  [Step 3] KL 발산
    0.571 (임계값: 0.2)
KL 발산이 크므로 업데이트를 억제한다.
  최종 정책  좋은 설명을 원해요다
✓ PPO는 정책 변화를 제어하여 안정성을 보장한다.


## 핵심 요약

| 알고리즘 | 방식 | 안정성 | 비용 |
|--------|-----|-------|-----|
| **REINFORCE** | 최고 응답 선택 | 낮음 | 낮음 |
| **PPO** | KL 제어 | 높음 | 중간 |
| **DPO** | 선호도 쌍 | 높음 | 높음 |
| **GRPO** | 그룹 상대 | 중간 | 높음 |

**결론**: 에이전트 메모리 + RL = 강화된 에이전트 시스템